## 환경설정

In [9]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive

gauth = GoogleAuth()
gauth.LocalWebserverAuth()  # 처음 실행 시 브라우저 인증
drive = GoogleDrive(gauth)

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=1071344378062-kv36e6mgl1vnr89evq958br79d063h8p.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=offline&response_type=code

Authentication successful.


In [3]:
import os

google_dirve_rt_id = os.getenv("GOOGLE_DRIVE_RT_ID")

In [4]:
playlist_id = "PLGiaCgd9PatcGBfZ7xTGdTAsHoNPRQ_AP"
collection_name = "content-250623"

## 문서 로드

In [5]:
import json

# 파일 탐색
file = drive.ListFile({
    'q': f"'{google_dirve_rt_id}' in parents and title = '{playlist_id}.json' and trashed = false"
}).GetList()

# 파일 로드
if file:
    contents = json.loads(file[0].GetContentString())

len(contents), type(contents)

(237, dict)

In [6]:
from langchain_core.documents import Document

# 타입 변환
documents = []
for content_id, data in contents.items():
    metadata = {
        k: ','.join(v) if isinstance(v, list) else v
        for k, v in data.items()
        if k != 'content'
    }
    doc = Document(
        page_content=data['content'], 
        metadata=metadata,
        id=content_id
    )
    documents.append(doc)

    print(f'{doc.id} : {doc.metadata.get('title')}, {len(doc.page_content)}')

pMevsZErXr4 : (ENG/VIET/FR SUB) 제653회 사주 팔자, 피할 수 없는 숙명이 정말 있는 건가요?, 2335
sJv1OULYFiI : 괴로움도 마음의 습관이다, 3333
sIkjsuLCNMM : 자존감과 열등감 (모음), 4481
PpPSpweXXFk : 일상에서 행복찾기2, 8342
Sht7uMLcO90 : [법륜스님의 즉문즉설 제1320회] 초혼, 재혼이 다 힘들고 자식들도..., 1963
x0zGJGG1xsc : 위기의 노부부, 행복한 소통법, 3446
OoEWGbUK62c : 부모를위한 즉문즉설 제1편 아이 잘 키우는 법 (모음), 7055
aakjPrxX7hw : 열심히 살았으나 결과는 참담합니다, 9309
Yt3NhEHazuQ : 일상에서 행복찾기 1 (모음), 5528
Zx3b1O-IxHI : (EN-SUB) 나보다 더 힘든 사람, 12986
Vqi3YtVCDTo : 남편과 항상 같이 있어도 행복한 법, 2162
l3Wim1oPlUY : 일상에서 행복찾기 6, 4991
BmcCGU8eb_s : (ENG/中文- SUB)  1207회 마음이 힘들 때 우울증 극복하는 방법, 1595
xPXr5oBwUyk : (EN-SUB)627회.자유롭고 행복한 인간 관계(법문), 2079
DJwCCAx8S4w : 제가 전생에 무슨 죄가 있어 이 고생을 할까요?, 2143
YnL7MQsZPYw : [법륜스님의 즉문즉설 1253회] 유산을 아들에게 물려주고 싶은데 딸이 걸려요, 11401
RpaLcuN8lwQ : 이혼, 해도 고민 안 해도 고민, 4483
sH1mkFDXRN4 : [법륜스님의 즉문즉설 제 1497회] 명절 차례와 제사는 꼭 지내야 되는 건지요?, 1550
XcTUeEZOj2s : (EN/日本語/FR SUB) 제721회 진정한 부부간의 사랑 (가정생활:법문), 2849
DqWSPzqA_C8 : [법륜스님의 즉문즉설 제 1439회] 남편과의 다툼, 2867
MsyJxUu-yWE : 장모와 사위, 가깝고도 먼 사이, 9809
oZvPb2

In [7]:
from transformers import GPT2TokenizerFast

# 긴 문서 확인
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
for doc in documents:
    token_len = len(tokenizer.encode(doc.page_content))

    if token_len > 30000:
        print(f"id={doc.id}, tokens={token_len}")

c:\Users\CNXK\Documents\mind_lantern_s\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Token indices sequence length is longer than the specified maximum sequence length for this model (4785 > 1024). Running this sequence through the model will result in indexing errors


## 문서 인덱싱

In [8]:
from langchain_openai.embeddings import OpenAIEmbeddings

# 임베딩 정의
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536
)

# 문서 임베딩
# document_embeddings = embeddings.embed_documents(
#     [chunk.page_content for chunk in docs]
# )

In [12]:
from langchain_pinecone import Pinecone
from uuid import uuid4

# 벡터 스토어 로드
vectorstore = Pinecone(
    embedding=embeddings,
    index_name="mind-lantern"
)

# 벡터 스토어 저장
docs_len = len(documents)
batch_size = 10
uuids = [str(uuid4()) for _ in range(docs_len)]
for i in range(0, docs_len, batch_size):
    vectorstore.add_documents(
        documents=documents[i:i+batch_size],
        ids=uuids[i:i+batch_size]
    )
    print(f"--- {i} 저장 ---")


--- 0 저장 ---
--- 10 저장 ---
--- 20 저장 ---
--- 30 저장 ---
--- 40 저장 ---
--- 50 저장 ---
--- 60 저장 ---
--- 70 저장 ---
--- 80 저장 ---
--- 90 저장 ---
--- 100 저장 ---
--- 110 저장 ---
--- 120 저장 ---
--- 130 저장 ---
--- 140 저장 ---
--- 150 저장 ---
--- 160 저장 ---
--- 170 저장 ---
--- 180 저장 ---
--- 190 저장 ---
--- 200 저장 ---
--- 210 저장 ---
--- 220 저장 ---
--- 230 저장 ---


In [18]:
from pprint import pprint

doc = vectorstore.similarity_search(" ", k=1)[0]
pprint(doc.id)
pprint(doc.metadata.get('title'))
pprint(doc.page_content)

'afcffdb2-bf48-4e26-8230-8069e63262b8'
'(EN-SUB)인간관계 시리즈2 외로움과 상처'
('질문자: 그런데 둘 다 하고 싶어요. 술 마시는 것도 좋아하고, 멋진 사람도 만나고 싶어요. 그럼 술 마시고 외로워하고, 멋진 사람 만나 '
 '위축되면 되나요? 좀 더 자연스럽게 인간관계를 맺고 싶어요.\n'
 '스님: 자연스럽게 인간관계를 맺고 싶으면 자연스럽게 맺으면 되잖아요.\n'
 '질문자: 그런데 되게 겁나요.\n'
 '스님: 그럼 겁 좀 내면 되잖아요. 뭐가 문제라고? "걷고 싶은데 다리 아파요" 하면 "다리 아픈 것 각오하고 걸으세요" 하는 것과 '
 '같아요. 제가 "스님 노릇 힘들어요. 졸리고, 무릎 아프고, 허리 아프고, 목 아프고, 외로워요" 하면 질문자는 뭐라고 할 건가요?\n'
 '질문자: 그래라.\n'
 '스님: (웃음) "그럼 그만두면 될 거 아니에요?" 하겠죠. 그런데 몇십 년 스님만 했는데 그만두면 뭘 하겠어요? "그럼 계속하세요" 할 '
 '거 아니에요? "밥 먹고 싶은데 배불러요" 하는 것과 같아요. "술 먹으면 외로워요" 하면 "술 안 먹으면 되잖아요." "먹고 싶어요" '
 '하면 "먹으면 되잖아요." "외로운데요" 하면 "외로움 좀 타면 되잖아요." 뭐가 어렵나요?\n'
 '질문자: 그러게요. (웃음)\n'
 '스님: 별일 아닌 것을 문제 삼는 거예요. 저도 결혼, 육아, 연애 경험 없는데 자꾸 물어서 힘들다고 하면, "질문 받지 마세요" '
 '하겠죠? "그런데 자꾸 질문하는데요?" 하면 "강연 안 하면 되잖아요." "강연해달라고 하는데요?" 하면 "그래도 안 하면 되잖아요." '
 '"해달라는데 어떻게 안 해줘요?" 하면 "그럼 하면 되잖아요." 그걸 뭐 어떡해요?\n'
 '질문자: 그럼 감수하고 그냥 해요.\n'
 '스님: (웃음) 맞아요. 물건 사고 싶으면 돈이 들고, 돈 필요하면 일해야죠. "물건 사고 싶은데 돈이 없어요" 하면 안 사면 되고, '
 '"그래도 사고 싶은데 어떻게 해요?" 

## 검색 테스트

In [19]:
query = "불안해요."

retriever_k = vectorstore.as_retriever(
    search_kwargs={"k": 3},
)
retrieved_docs = retriever_k.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for doc in retrieved_docs:
    print(doc.metadata['title'])
    print('---'*20)
    print(doc.page_content[:100])
    print('==='*20)

쿼리: 불안해요.
검색 결과:
[법륜스님의 즉문즉설 제 1416회] 좌절감이 들때, 어떻게 해요
------------------------------------------------------------
스님: 안녕하세요. 우리가 불행한 이유는 크게 두 가지입니다. 첫째는 개인의 성격 때문인데, 성격이 급하거나 욕심이 많거나 자기주장이 강하면 불행해지기 쉽죠. 행복하려면 성격을 고
[법륜스님의 즉문즉설 1267회] 욕심을 버리고 사랑을 찾고 싶어요
------------------------------------------------------------
질문자: 안녕하세요, 스님. 지난 10년간 유학 생활을 하다 한국에 돌아왔는데, 유튜브로만 뵙다가 실제로 뵈니 너무 기쁘고 영광스럽습니다. 외국 생활 중 스님 말씀이 큰 힘이 되어
(EN/FR/VN-sub)There Are Too Many People I Dislike.싫은사람이 너무  많아서 고민이에요
------------------------------------------------------------
**질문자:** 싫은 사람이 너무 많아 고민입니다. 특히 싫은 사람이 잘 되는 걸 보면 배 아파요. 남편을 사랑하지만 시어머니 잔소리, 형님 돈 안 쓰는 것, 신랑 친구 와이프의 


In [21]:
query = "불안해요."

retriever_mmr = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 3,                 # 검색할 문서의 수
        'fetch_k': 8,           # mmr 알고리즘에 전달할 문서의 수 (fetch_k > k)
        'lambda_mult': 0.3,     # 다양성을 고려하는 정도 (1은 최소 다양성, 0은 최대 다양성을 의미. 기본값은 0.5)
        },
)
retrieved_docs = retriever_mmr.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for doc in retrieved_docs:
    print(doc.metadata['title'])
    print('---'*20)
    print(doc.page_content[:100])
    print('==='*20)


쿼리: 불안해요.
검색 결과:
[법륜스님의 즉문즉설 제 1416회] 좌절감이 들때, 어떻게 해요
------------------------------------------------------------
스님: 안녕하세요. 우리가 불행한 이유는 크게 두 가지입니다. 첫째는 개인의 성격 때문인데, 성격이 급하거나 욕심이 많거나 자기주장이 강하면 불행해지기 쉽죠. 행복하려면 성격을 고
결혼하기가 왜 이렇게 힘든가요
------------------------------------------------------------
**질문자1:** 남자친구가 부모님의 불행한 결혼 생활 때문에 본인도 결혼하면 불행해질 거라며 두려워해요. 정신과 치료까지 받았지만 오히려 역효과가 나서 저와 헤어지길 바라고 있습
[법륜스님의 즉문즉설 제 1497회] 명절 차례와 제사는 꼭 지내야 되는 건지요?
------------------------------------------------------------
질문자: 너무 반갑습니다. 명절 차례와 제사를 꼭 지내야 하는 건가요? 잘 지내면 자손이 잘 된다고 하고, 안 지내면 조상님께 죄송하다고 하는데, 정말 그런가요? 요즘엔 제사를 아


## 질의 테스트

In [22]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever_mmr,
    return_source_documents=True
)

# 질의 테스트
query = "불안해요."
result = qa_chain.invoke(query)
print(result)

{'query': '불안해요.', 'result': '불안하시군요. 스님께서는 불안감을 느끼는 분에게 다음과 같은 조언을 해주셨습니다.\n\n1.  **병원 진료:** 가장 좋은 방법은 병원에 가서 진료를 받는 것입니다. 스님께서는 불안증을 \'병\'으로 보셨으며, 옛날에는 정신과를 꺼렸지만 지금은 전문 분야이므로 가서 진료를 받으면 위험도를 줄일 수 있다고 하셨습니다. 병이기 때문에 "정신 차려라"고 한다고 나아지는 것이 아니라고 강조하셨습니다.\n\n2.  **수행 (절):** 완치를 위해서는 \'절\'을 하는 수행을 병행하는 것이 좋다고 하셨습니다. 절은 종교적인 의미를 넘어 \'내가 옳다\'는 고집을 내려놓는 몸의 표현이며, 겸손의 자세를 몸으로 익히는 것이라고 설명하셨습니다.\n\n3.  **기도문 암시:** 절을 하면서 "주님, 저는 주님의 은혜 속에서 편안하게 살고 있습니다. 감사합니다."라고 기도해 보라고 하셨습니다. 지금은 불안하더라도 계속 \'편안하다\'는 암시를 무의식에 주어 치료에 도움이 된다고 합니다. 절과 함께 이 기도를 하면 암시 효과가 훨씬 커진다고 하셨습니다.\n\n스님께서는 불안증이 어릴 때 트라우마에서 오는 경우가 많다고 하셨지만, 위에서 말씀드린 수행과 기도문은 원인이 무엇이든 상관없이 \'나는 편안하다\'고 계속 암시를 주어 치료에 도움이 된다고 강조하셨습니다.', 'source_documents': [Document(metadata={'channel': '법륜스님의 즉문즉설', 'duration': 988.0, 'like_count': 16251.0, 'tags': '즉문즉설,법륜스님,정토회,buddhism,pomnyun,깨달음,기도,수행,답답하면물어라,법륜,스님,Buddhahood,Gautama,Buddha,30대,남성,가정사,사회시사,인공지능,시사,본인,열등감,좌절감,사나이,자존감,주변환경,자기주장,의견존중', 'title': '[법륜스님의 즉문즉설 제 1416회] 좌절감이 들때, 어떻게 해요', 'upload_dat

In [23]:
from IPython.display import Markdown, display

display(Markdown(result['result']))

불안하시군요. 스님께서는 불안감을 느끼는 분에게 다음과 같은 조언을 해주셨습니다.

1.  **병원 진료:** 가장 좋은 방법은 병원에 가서 진료를 받는 것입니다. 스님께서는 불안증을 '병'으로 보셨으며, 옛날에는 정신과를 꺼렸지만 지금은 전문 분야이므로 가서 진료를 받으면 위험도를 줄일 수 있다고 하셨습니다. 병이기 때문에 "정신 차려라"고 한다고 나아지는 것이 아니라고 강조하셨습니다.

2.  **수행 (절):** 완치를 위해서는 '절'을 하는 수행을 병행하는 것이 좋다고 하셨습니다. 절은 종교적인 의미를 넘어 '내가 옳다'는 고집을 내려놓는 몸의 표현이며, 겸손의 자세를 몸으로 익히는 것이라고 설명하셨습니다.

3.  **기도문 암시:** 절을 하면서 "주님, 저는 주님의 은혜 속에서 편안하게 살고 있습니다. 감사합니다."라고 기도해 보라고 하셨습니다. 지금은 불안하더라도 계속 '편안하다'는 암시를 무의식에 주어 치료에 도움이 된다고 합니다. 절과 함께 이 기도를 하면 암시 효과가 훨씬 커진다고 하셨습니다.

스님께서는 불안증이 어릴 때 트라우마에서 오는 경우가 많다고 하셨지만, 위에서 말씀드린 수행과 기도문은 원인이 무엇이든 상관없이 '나는 편안하다'고 계속 암시를 주어 치료에 도움이 된다고 강조하셨습니다.